TODOs:
* Zero-Inflated: 3-pt field goals made, 3-pt field goals attempted, pnr handlers made, pnr handlers attempted, 
* Non-Gaussian: blocks, screen assists, points off screen assists, screens off made, screens off attempted, posts up made, posts up attemtpted, isolations made, isolations attempted, hand offs made, hand offs attempted, cuts made, cuts attempted, pnr handlers made, pnr handlers attempted, pnp made, pnp attempted, opp pick-n-roll shots, opp pick-n-roll shots made, opp pick-n-pop shots, opp pick-n-pop shots made, drives made, drives with shot, right drives, left drives, right drives made, left drives made
*  (FIXED) What's up with 3-pt data? Very low eFG(%) for some players

Requires following packages to be installed with anaconda:
seaborn, pandas, openpyxl, jupyter (conda-forge), numpy, sklearn

In [2]:
import pandas as pd
import numpy as np
import os
import re

In [18]:
# Minutes in data are displayed e.g. 12:34, which is 12 minutes and 34 seconds. This function converts it to a float
def minute_str_to_float(time):
    minutes = int(time.split(":")[0])
    seconds = int(time.split(":")[1])
    return minutes + seconds / 60

# Same entries are of the form 60%, this function converts it to a float like 0.6
def convert_percentage(percentage):
    if percentage == "-":
        return 0
    return float(percentage.strip("%")) / 100

def load_data() -> pd.DataFrame:
    # Extract League and Season information from filename
    regex = r".+\. (.+) \- (\d{4}-\d{4}).xls$"
    frames = []
    s = 0
    for file in os.listdir("data_raw"):
        filename = os.fsdecode(file)
        m = re.match(regex, filename)
        if m is not None:
            s += 1
            # Test something different
            df = pd.read_excel(os.path.join("data_raw", filename), sheet_name="Box score", engine="openpyxl")
            # Add information contained in filename
            df.insert(0, "Season", [m.group(2)] * len(df), False)
            df.insert(0, "League", [m.group(1)] * len(df), False)
            print(f"Processed {m.group(1)} - {m.group(2)}")
            frames.append(df)
    
    df = pd.concat(frames, ignore_index=True)
    df = df.rename(columns={"Unnamed: 0": "Jersey number", "Unnamed: 1": "Player name", "Unnamed: 2": "Team name"})
    df.columns = df.columns.str.lower()
    # df.replace("-", "0", inplace=True)
     # convert minutes string to float
    df["minutes"] = df["minutes"].apply(minute_str_to_float)
    for column in df.columns.tolist():
        if "%" in column or "percentage" in column:
            df[column] = df[column].apply(convert_percentage)

    return df

# Load data to dataframe
df = load_data()

Processed Liga Endesa - 2020-2021
Processed BETCLIC Elite - LNB Pro A - 2022-2023
Processed Liga Endesa - 2021-2022
Processed Basket League - 2022-2023
Processed LEB Oro - 2020-2021
Processed LEB Oro - 2021-2022
Processed Basketball Bundesliga - 2022-2023
Processed LEB Plata - 2022-2023
Processed FIBA Champions League - 2020-2021
Processed FIBA Champions League - 2021-2022
Processed Basket League - 2020-2021
Processed LEB Oro - 2022-2023
Processed Basket League - 2021-2022
Processed Liga Endesa - 2022-2023
Processed BETCLIC Elite - LNB Pro A - 2021-2022
Processed BETCLIC Elite - LNB Pro A - 2020-2021
Processed FIBA Champions League - 2022-2023
Processed LEB Plata - 2020-2021
Processed Basketball Bundesliga - 2020-2021
Processed LEB Plata - 2021-2022
Processed Basketball Bundesliga - 2021-2022


The below function shows that there are 14 duplicate entries out of a total of about 7400 data points. In addition, most of them didn't play any significant minutes which is why it is reasonable to just drop them.

We observe in a first step that quite many players have played in their national league as well as in one of the European competitions. In a next step we checked if players played in different leagues within the same season. We observe only two players: Sasa Borovnjak and Simas Jarumbauskas. Both playing in the LEB Oro (Spanish first league).

After these observations we need to aggregate the stats of these players by a weighted average (weighted by minutes played).

In [19]:
# Check for duplicate entries for columns "player name", "team name", "league", "season"
# Returns a boolean series indicating if the row is a duplicate
duplicates_indicator = df.duplicated(subset=["player name", "team name", "league", "season"], keep=False)
# duplicates_all = df[duplicates_indicator].sort_values(by=["player name", "team name", "league", "season"])
# duplicates = df[duplicates_indicator][["player name", "team name", "league", "season"]].sort_values(by=["player name", "team name", "league", "season"])

# Drop duplicates and perform sanity check
print(f"Num duplicates before drop: {duplicates_indicator.sum()}")
df = df[duplicates_indicator == False]
print(f"Sanity check for num of duplicates:  {df.duplicated(subset=['player name', 'team name', 'league', 'season'], keep=False).sum()}")

Num duplicates before drop: 14
Sanity check for num of duplicates:  0


We consider players playing in multiples seasons as different data points. Also, we don't want to consider the lower (Spanish) leagues for our clustering algorithms. To performs some exploratory analysis we drop some of the meta information like jersey number, team name, season, player name. We proceed by performing some univariate analysis and creating some plots to investigate the relationship between the features. 

In [27]:
# Check how many players have played for multiple teams in the same season
df_check_ind = df.duplicated(subset=["player name", "season"], keep=False)
df_multiples = df[df_check_ind]

# In df_multiples, look for rows where player is the same but team name is different
grouped = df_multiples.groupby(['player name', 'season'])

# Filter groups where the player is in more than one team
filtered_groups = grouped.filter(lambda x: x['team name'].nunique() > 1).sort_values(by=['player name', 'season'])
filtered_groups

,league,season,jersey number,player name,team name,games played,minutes,points,points per player's possession,field goals made,...,"drives, %",right drives made,right drives,"right drives made, %",left drives made,left drives,"left drives made, %",opp drives shots made,opp drives shots,"opponent drives shots made, %"
2105,LEB Plata,2022-2023,3,Aaron Ganal,Talavera,18,18.616667,4.4,0.64,1.61,...,0.12,0.15,0.85,0.18,-,0.38,0.00,0.62,0.92,0.67
4248,LEB Oro,2022-2023,11,Aaron Ganal,BC MoraBanc Andorra,1,10.900000,-,-,-,...,0.00,-,-,0.00,-,-,0.00,-,-,0.00
271,Liga Endesa,2020-2021,34,Aaron Jones,Bilbao Basket,8,19.850000,4.1,0.76,1.62,...,0.32,0.12,0.25,0.48,-,0.12,0.00,0.62,0.88,0.70
2748,FIBA Champions League,2020-2021,34,Aaron Jones,Cholet Basket,5,25.450000,8,1.05,3.2,...,0.50,0.2,0.4,0.50,-,-,0.00,0.6,1,0.60
5221,BETCLIC Elite - LNB Pro A,2020-2021,34,Aaron Jones,Cholet Basket,28,21.816667,8,0.97,3.2,...,0.62,0.07,0.14,0.50,0.11,0.14,0.79,0.61,1.07,0.57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1553,LEB Oro,2021-2022,1,Yannick Kraag,CB Prat,33,22.816667,9,0.95,3.7,...,0.56,0.27,0.52,0.52,0.36,0.64,0.56,0.91,1.88,0.48
320,Liga Endesa,2020-2021,55,Zsombor Maronka,Club Joventut Badalona,2,3.016667,2,-,1,...,0.00,-,-,0.00,-,-,0.00,0.5,0.5,1.00
6258,LEB Plata,2020-2021,5,Zsombor Maronka,CB Prat,3,21.950000,6,-,2,...,0.00,-,-,0.00,-,0.33,0.00,1,2.3,0.43
852,Liga Endesa,2021-2022,55,Zsombor Maronka,Club Joventut Badalona,17,4.966667,1.06,0.68,0.47,...,1.00,-,-,0.00,0.06,0.06,1.00,0.06,0.12,0.50


Filtered groups show that 566 players played for multiple teams within the same season, sometimes even across different leagues. This can happen if switching to a different league within the same season. Because this is such a large number we don't see any reason to manually find the players where either the team or competition is constant across duplicate entries. Hence, as done below, we set these values to "aggregated".

In [5]:
# If a player appears in multiple dataframes, aggregate the data, i.e. weighted average by seconds played

df_check_ind = df.duplicated(subset=["player name", "season"], keep=False)
df_multiples = df[df_check_ind]
print(f"Check for multiple entries (before): {df_multiples.shape[0]}")


def aggregate_stats(df, df_multiples):
    # After aggregating a player should not appear multiple times within the same season
    # Iterate over each player, season pair
    # df.replace("-", np.nan, inplace=True)
    df.replace("-", 0, inplace=True)
    df.replace(np.nan, 0, inplace=True)

    # This will produce a warning "treating keys as positions deprecated". But it isn't an issue because we are using
    # the index simply as a loop counter.
    for i, (player_name, season) in enumerate(df_multiples[["player name", "season"]].drop_duplicates().values):
        # Extract all rows for the player, season pair
        df_player = df[(df["player name"] == player_name) & (df["season"] == season)]
        # First compute minutes player per season
        total_minutes = df_player["minutes"] * df_player["games played"]
        total_across_all = total_minutes.sum()
        weights = total_minutes / total_across_all
        df_player_new = df_player.copy()
        # Not sure if the below line of code is still necessary
        df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].apply(pd.to_numeric, errors='coerce')
        # Multiply each columns by the weights and sum them up. Apply the weights row-wise to indicate
        # the importance of different statistics (weighed by minutes played)
        df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
        # We now have the aggregated data for the player, season pair. Each row contains the the aggregated dat.
        # So we need to drop duplicates.
        df_player_new = df_player_new.drop_duplicates(subset=["player name", "season"])
        # Still need adjust the games played and minutes played. Also because the leage and team data are now ambiguous, 
        # we need to set them to "aggregated".
        total_games = df_player["games played"].sum()
        avg_minutes = df_player["minutes"].mul(pd.Series(df_player["games played"]), axis=0).sum() / total_games
        df_player_new["games played"] = total_games
        df_player_new["minutes"] = avg_minutes
        df_player_new["league"] = "aggregated"
        df_player_new["team name"] = "aggregated"

        # Now, df_player_new contains the aggregated data for the player, season pair (only one row).
        # Need to copy it back to the original dataframe and drop the rows that were aggregated
        df.drop(df[(df["player name"] == player_name) & (df["season"] == season)].index, inplace=True)
        # Append the aggregated data
        df = pd.concat([df, df_player_new], ignore_index=True)
        length = len(df_multiples[["player name", "season"]].drop_duplicates())
        if i % 50 == 0:
            print(f"Aggregated data for {i}/{length} players.")

    return df

df = aggregate_stats(df, df_multiples)

# Perform sanity check, i.e. check if there are still duplicates. That is, there should only exist one player, season pair
df_check_ind = df.duplicated(subset=["player name", "season"], keep=False)
df_mulitiples = df[df_check_ind]
print(f"Check for multiple entries (after): {df_mulitiples.shape[0]}")

Check for multiple entries (before): 1678
Aggregated data for 0/818 players.


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys a

Aggregated data for 50/818 players.


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys a

Aggregated data for 100/818 players.


...

/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys a

Aggregated data for 800/818 players.


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys a

Check for multiple entries (after): 0


/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_player_new[df_player_new.columns[7:]] = df_player_new[df_player_new.columns[7:]].mul(weights, axis=0).sum(axis=0)
/var/folders/76/8gdl7zm95b3254dfqr95r58w0000gn/T/ipykernel_12238/2125322419.py:29: FutureWarning: Series.__getitem__ treating keys a

In [12]:
df[df["league"] == "aggregated"]

,league,season,jersey number,player name,team name,games played,minutes,points,points per player's possession,field goals made,...,"drives, %",right drives made,right drives,"right drives made, %",left drives made,left drives,"left drives made, %",opp drives shots made,opp drives shots,"opponent drives shots made, %"
5746,aggregated,2020-2021,22,Emir Sulejmanovic,aggregated,53,13.699057,4.877660,1.102305,2.011631,...,0.448936,0.052376,0.089787,0.433971,0.037411,0.052376,0.531241,0.264965,0.487340,0.547624
5747,aggregated,2020-2021,19,Giorgi Shermadini,aggregated,54,23.444444,17.000000,1.265275,5.875458,...,0.562343,0.034725,0.072454,0.464247,0.034725,0.057363,0.547251,0.520988,0.961355,0.534175
5748,aggregated,2020-2021,34,Tyler Cavanaugh,aggregated,54,22.920679,8.255228,1.197657,2.927614,...,0.476359,0.090418,0.150000,0.605271,0.164895,0.406820,0.412971,0.497239,0.932091,0.548745
5749,aggregated,2020-2021,35,Francisco Guerra,aggregated,54,14.474074,7.736097,1.130415,2.873610,...,0.683195,0.073751,0.129029,0.544864,0.040556,0.040556,1.000000,0.165693,0.555420,0.320829
5750,aggregated,2020-2021,6,Bruno Fitipaldo,aggregated,55,23.051515,9.431313,0.881470,3.172525,...,0.458530,0.577732,1.442525,0.388722,0.455687,0.820288,0.564025,0.672747,1.401182,0.480000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,aggregated,2020-2021,9,Rihards Lomazs,aggregated,27,24.194444,15.609108,1.120195,4.782732,...,0.463046,0.420065,0.938078,0.445033,0.297215,0.603436,0.441319,0.701384,1.292573,0.529137
6560,aggregated,2020-2021,24,Rasid Mahalbasic,aggregated,44,20.975758,10.726524,0.987776,4.276936,...,0.547183,0.498040,0.888468,0.547183,0.069061,0.112224,0.535222,0.344613,0.907694,0.384184
6561,aggregated,2020-2021,19,Ikenna Iroegbu,aggregated,8,20.275000,8.064529,0.753387,2.425812,...,0.319359,0.399199,0.516132,0.532265,0.000000,0.500000,0.000000,1.000000,1.733868,0.590485
6562,aggregated,2020-2021,4,Edgar Sosa,aggregated,18,26.897222,16.275534,1.072613,5.282660,...,0.629287,0.843064,1.375772,0.648860,0.379264,0.657957,0.567387,0.940143,1.498979,0.625653


We now drop some of the data for analysis. We observed that the statistic for "deflections" has not been gathered for every league. In addition, the "usage percentage" is zero for all players. Hence, we remove these two features. We also drop every data point of players which do not play in a first league, and as well players which have not played a sufficient amount. To this end, we define a cutoff of a minimum amount of minutes a player must have played in a specific season.

In [13]:
"player name" in df.columns

True

In [14]:
CUTOFF = 230

print(f"Data points before: {len(df)}")

df_pre_transform = df.copy()
df_pre_transform = df_pre_transform.drop(labels=["deflections", "usage percentage"], axis=1)
# Keep old df to compare to reduced one. Check if the below transformations worked!

def fix_effective_field_goal(df):
    # Assumes that FGA is non-zero
    # eFG percentage is computed as (FG + 0.5 * 3P) / FGA
    # First, find data where eFG is equals to zero
    zero_efg_ind = df['effective field goal percentage'] == 0
    # Ensure that we don't divide by zero
    zero_efg_df = df[zero_efg_ind & (df['field goals attempted'] != 0)].copy()
    # Update orignal df with the new eFG values, but only where eFG is zero
    df.loc[zero_efg_ind, 'effective field goal percentage'] = (zero_efg_df['field goals made'] + 0.5 * zero_efg_df['3-pt field goals made']) / zero_efg_df['field goals attempted']
    # For sanity check: print out number of zeros in effecitve field goal column
    print(f"Number of zeros in eFG column: {df.loc[df['effective field goal percentage'] == 0, 'effective field goal percentage'].sum()}")


def drop_too_little_data(df, cutoff):
    """
    df is the dataframe containing all the data over multiple seasons and multiple leagues.
    cutoff is the minimum number of minutes a player must have played in order to be considered for analysis.
    """
    # Drop rows of players having played less than e.g. 230 minutes, i.e. games played times seconds per game
    df.drop(df[df["games played"] * df["minutes"] < cutoff].index, inplace=True)
 
    
def store_normalization_csv(df):
    minutes = pd.Series(df["minutes"])
    num_possesions = pd.Series(df["number of player's possessions"])
    minutes.to_csv("minutes.csv", index=False)
    num_possesions.to_csv("num_possessions.csv", index=False)



low_leagues = ["LEB Plata", "Liga Endesa"]
def drop_low_leagues(df):
    # Drop all rows, i.e. players, where the league is either "LEB Plata" or "Liga Endesa"
    mask = -df["league"].isin(low_leagues)
    df = df[mask]

# !!CHANGED!!: now also include lower leagues
# drop_low_leagues(df_pre_transform)
# Maybe change this
drop_too_little_data(df_pre_transform, CUTOFF)
# Call this function after the above to make sure we remove data where FG attempted is 0. Would lead to division by 0.
fix_effective_field_goal(df_pre_transform)
store_normalization_csv(df_pre_transform)

Data points before: 6564
Number of zeros in eFG column: 0.0


We now deal with the features which don't seem to follow a Gaussian distribution. These were already listed above but are here for convenience again:

blocks, screen assists, points off screen assists, screens off made, screens off attempted, posts up made, posts up attemtpted, isolations made, isolations attempted, hand offs made, hand offs attempted, cuts made, cuts attempted, pnr handlers made, pnr handlers attempted, pnp made, pnp attempted, opp pick-n-roll shots, opp pick-n-roll shots made, opp pick-n-pop shots, opp pick-n-pop shots made, drives made, drives with shot, right drives, left drives, right drives made, left drives made

We have two normalization candidates: by "minutes" or by "number of player's possessions". We decided on the following:
* For offensive data: normalize by possessions
* For other data: normalize by minutes

However, data points which inherintly already include the number of possessions in their metric should not be normalized. These include:
* 'points per possession', 'offensive rating', 'true shooting percentage' (indirectly), 'field goals, %', '3-pt field goals, %', 'free throws, %', '2-pt field goals, %'

### Justification for Using Yeo-Johnson Transformation (ChatGPT)

The Yeo-Johnson transformation is applied to several features in this dataset that exhibit exponential-like distributions. This transformation is essential for the following reasons:

1. **Normalize Skewness**: Many machine learning algorithms, including clustering techniques like k-means, assume that the features follow a normal distribution. The Yeo-Johnson transformation helps in correcting skewness and stabilizing the variance of the data, making these assumptions more tenable.

2. **Handle Zero and Negative Values**: Unlike other transformations that require strictly positive values (e.g., Box-Cox), the Yeo-Johnson transformation can be applied to data with zero or negative values, increasing its applicability to our dataset without the need for data adjustments or exclusions.

3. **Enhance Clustering Efficacy**: By transforming skewed features to resemble a normal distribution, the transformed features contribute more effectively to the Euclidean distance calculations used in k-means clustering. This is critical as skewed data can distort distance measures and lead to suboptimal clustering results.

4. **Uniformity and Comparability**: Transforming the features ensures that they are on a more comparable scale, which is crucial when combining multiple features in clustering algorithms. This uniformity can lead to more meaningful and interpretable clusters.

By applying the Yeo-Johnson transformation, we prepare the data in a way that enhances the overall performance of the subsequent clustering process and ensures that the feature scales do not unduly influence the model's outcomes.


In [15]:
from sklearn.preprocessing import PowerTransformer

NORMALIZATION_METHOD = "number of player's possessions"
NON_GAUSSIAN = ['blocks', 'screen assist', 'points off screen assists', 'screens off made', 'screens off attempted',
              'posts up made', 'posts up attempted', 'isolations made', 'isolations attempted', 'hand off made',
              'hand off attempted', 'cuts made', 'cuts attempted', 'pnr handlers made', 'pnr handlers attempted',
              'pnp made', 'pnp attempted', 'opp pick-n-roll shots', 'opp pick-n-roll shots made', 'opp pick-n-pop shots',
              'opp pick-n-pop shots made', 'drives made', 'drives with shot', 'right drives', 'left drives', 
              'right drives made', 'left drives made'

]

# Normalize the data by minutes played before or after applying the log-transformation? Probably apply transformation first.
# Also, better to normalize by number of player possesions or by minutes played?
# 

def apply_log_transformation(df, features):
    """
    Assumption: data for low leagues already dropped and data for players with insuficient minutes played also dropped.
    df: dataframe containing all the data
    features: list of features to be log-transformed
    """
    # add 1 to avoid log(0)
    for feature in features:
        df[feature] = np.log(df[feature] + 1)

def apply_power_tranform(df, features):
    pt = PowerTransformer(method='yeo-johnson')
    for feature in features:
        # Use double brackets because fit_transform expects a 2D array
        df[feature] = pt.fit_transform(df[[feature]])
    return df

# !!CHANGED!!: removed the line which dropped the first seven columns, i.e. now only normalizing and removing no features
def normalize_features(df, normalization="minutes"):
    """
    normalize: either "minutes", or "number of player's possessions", or None
    """
    # Normalize all columns by minutes played
    minutes = pd.Series(df["minutes"])
    num_possesions = pd.Series(df["number of player's possessions"])
    # Before dropping irrelevant features, all player names to csv file
    player_names = df["player name"]
    player_names.to_csv("player_names.csv", index=False)
    if normalization is not None:
        print(f"Normalized data by {normalization}.")
        # Assumes that we have no zero entries.
        if normalization == "minutes":
            # TODO: think about which features should not be normalized by minutes
            df = df.div(minutes, axis=0)
        else:
            # Normalize all features except "offensive rating", "true shooting percentage", and "points per player's possessions". 
            # These should be kept as they are.
            # !!CHANGED!!: added more percentage stats which should not be normalized, the rest of the percentage will be removed
            # Also the metadata like league and season can't be normalized because they are strings
            keep = ["league", "season", "jersey number", "player name", "team name", "offensive rating", "true shooting percentage", 
                    "points per player's possession", 'field goals, %', '3-pt field goals, %', 'free throws, %', '2-pt field goals, %']
            df_temp = df.copy()
            df_temp = df_temp.drop(labels=keep, axis=1)
            # Normalize all columns by number of player's possessions
            df_temp = df_temp.div(num_possesions, axis=0)
            # Add dropped columns back to the dataframe by creating a mask for features that were normalized and overriding it with df_temp
            mask = ~df.columns.isin(keep)
            df.loc[:, mask] = df_temp
        return df
    else:
        print("No normalization performed.")
        return df

# df_post_transform = apply_log_transformation(df_pre_transform, NON_GAUSSIAN)

# !!CHANGED!!: now also normalizing the post_transformed data
df_post_transform = apply_power_tranform(df_pre_transform, NON_GAUSSIAN)
df_post_transform = normalize_features(df_post_transform, normalization=NORMALIZATION_METHOD)
## Export to csv file s.t. it can be later used for the ML
#if NORMALIZATION_METHOD is None:
#    df_new.to_csv(f'preprocessed_data.csv', index=False)
#else:
#    df_new.to_csv(f"preprocessed_normalized_{NORMALIZATION_METHOD}.csv", index=False)
#print(f"Data points left: {len(df_new)}")

Normalized data by number of player's possessions.


#### Discussion of what histograms showed (Copy from "explore_data.ipynb"):

* Many approximately normally distributed variables and many which seem to follow an exponential (or similar) distribution.
* Quite a few features seem to have missing values. This can be seen in the histograms which follow a distribution but for the max/min values there is a spike in the plot
* We can try different data imputation methods and visually inspect that these do not significantly alter the distributions.
* Generally, the "percentage" features always seem kind of redundant because they depend on the total attempts and attempts made anyways. Might just remove them.

Due to the reasons stated above, the following features have been removed (percentage and "many" insignificant number of attempts). These are specialized shooting/scoring statistics and thus have smaller sample size:
* "(un)contested fields goald, %", "opponent's field goals, %", "transition attacks, %", 
* "catch and shoot shots made, %", "catch and drive shots made, %", "screens off, %", "post up, %", "isolations, %", "hand off, %", 
* "cuts, %", "pr handler, %", "pr roller, %", "pick-n-pops, %", "opponent transition shots made, %", "opp catch and shoot shots made, %", 
* "opp catch and drive shots made, %", "opp screens off shots made, %", "opponent post up shots made, %", 
* "opponent isolation shots made, %", "opponent hand off shots made, %", "opponent cuts shots made, %", "opponent pick-n-roll shots made, %", 
* "opponent pick-n-pop shots made, %", "drives, %", "right drives made, %", "left drives made, %", "opponent drives shots made, %"

The above histograms are stored in "./histograms" and the file "histogram_set_4.png" shows that we have very limited data for pick-n-roll (roller). Hence, we just remove them altogether. These additionally include:
* "pnr rollers made", "pnr rollers attempted"
* For now we still include the "handlers" as there is a little bit more data available for them.

After removing these features it seems like data imputation might not be necessary. While still many features have predominantly many (close to) zero values, this might be very reasonable to assume. E.g. many players do not attempt any 3-pt shots. These would then correspond to zero-inflated features and we might have to deal with them separately. 

In [16]:
# Features still include data like name, league etc. bc they are needed for the following code
# Should be remembered that these should be not included in the ML model

# !!CHANGED!!: added this piece of code from "explore_data.ipynb" which removes the percentage features
REMOVED_FEATURES = ["contested field goals, %", "uncontested field goals, %", "opponent's field goals, %", "transition attacks, %", 
                  "catch and shoot shots made, %", "catch and drive shots made, %", "screens off, %", "post up, %", "isolation, %", "hand off, %", 
                  "cuts, %", "pr handler, %", "pr roller, %", "pick-n-pops, %", "opponent transition shots made, %", "opp catch and shoot shots made, %",
                  "opp catch and drive shots made, %", "opponent screens off shots made, %", "opponent post up shots made, %",
                  "opponent isolation shots made, %", "opponent hand off shots made, %", "opponent cuts shots made, %", "opponent pick-n-roll shots made, %",
                  "opponent pick-n-pop shots made, %", "drives, %", "right drives made, %", "left drives made, %", "opponent drives shots made, %",
                  "pnr rollers made", "pnr rollers attempted"]

df_post_transform = df_post_transform.drop(REMOVED_FEATURES, axis=1)
# Count how many features still have "%" in their name and list them
print(df_post_transform.columns[df_post_transform.columns.str.contains('%')].tolist())
print(f"Number of features left: {len(df_post_transform.columns)}")

['field goals, %', '3-pt field goals, %', 'free throws, %', '2-pt field goals, %']
Number of features left: 100


In [122]:
last_season_filter = df_post_transform[["player name", "season"]].groupby("player name").max().reset_index()

In [123]:
# This is a terrible way to do it
indices = []
for elem in last_season_filter.iterrows():
    indices.append((df_post_transform[(df_post_transform["player name"] == elem[1]["player name"]) &
                             (df_post_transform["season"] == elem[1]["season"])].index[0]))

In [124]:
most_recent_season = df_post_transform.loc[indices]
most_recent_season.to_csv("train_data_yeo_new2.csv")

In [125]:
# Sanity check, sc should be empty
sc = pd.DataFrame(most_recent_season.groupby("player name").count()["season"]).reset_index()
sc[sc["season"] != 1]

,player name,season
